In [16]:
import requests  # [+] HTTP 요청
from bs4 import BeautifulSoup  # [+] 웹스크래핑
import pandas as pd

In [54]:
# 1. User-Agent 설정
# 우리가 실제 크롬 브라우저를 사용하는 일반 사용자인 것처럼 서버를 속이는 '위장 신분증' 역할
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

search_keyword = '이란'  # [+] 검색 키워드
url = f'https://search.naver.com/search.naver?where=news&query={search_keyword}'  # 검색한 웹페이지 주소

In [56]:
# 2. 페이지 요청 및 파싱
res = requests.get(url, headers=headers)  # [+] 네이버 서버에 페이지 정보 요청(GET)
res.raise_for_status()  # 서버가 응답을 잘 주었는지 확인(정상 200, 페이지 없음 404 등)
soup =  BeautifulSoup(res.text, 'html.parser') # [+] BeautifulSoup 객체 생성

In [34]:
res

<Response [200]>

In [58]:
# 3. 질문자님이 찾은 클래스로 타겟팅 (띄어쓰기를 '.'으로 변경)
# [+] 찾고자 하는 HTML 태그 정의
target_class = "span.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1"
title_spans = soup.select(target_class) # [+] 조건에 맞는 모든 태그를 찾아 리스트로 저장

extracted_data = []

In [44]:
title_spans

[<span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">Genome &amp; Co. Announces Technology Transfer Deal with <mark>Ellipsis</mark> Pharma for ...</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">The <mark>Ellipsis</mark> in the Title of Tarantino’s New Film Is Explained … Sort ...</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">Cellumed Hits Upper Limit on Hopes for Ownership Sale[K-Bio Pulse]</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">[더벨][클리니컬 리포트] 지놈앤컴퍼니, PD-1 장벽의 해법 'CNTN4' ADC</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1">At Dance Reflections, Embodied Acts of Memory</span>,
 <span class="sds-comps-text sds-comps-text-ellipsis sds-comps-text-e

In [60]:
# 4. 데이터 추출
for span in title_spans:
    # [+] 기사 제목 추출
    title = span.text.strip()
    
    # [+] 기사 링크 추출: 제목(span)을 클릭 가능하게 만드는 부모 <a> 태그를 역추적하여 찾음
    parent_a_tag = span.find_parent('a')
    link = parent_a_tag['href'] if parent_a_tag and parent_a_tag.has_attr('href') else "링크 없음"
    
    extracted_data.append({
        '제목': title,
        '링크': link
    })

In [62]:
# 5. [+] DataFrame으로 변환 및 확인
df = pd.DataFrame(extracted_data)
print(f"총 {len(df)}개의 기사를 추출했습니다.\n")
display(df)

총 10개의 기사를 추출했습니다.



,제목,링크
0,[단독] 이란에 26척 정보 넘긴 韓구출작전…‘역봉쇄’ 변수 만났다,https://www.joongang.co.kr/article/25419917
1,"미·이란, 물밑 대화 진행중…2차 대면협상 가능성도 준비",https://www.yna.co.kr/view/AKR2026041402030000...
2,"""미국-이란 협상단, 이번주 후반 파키스탄서 만난다""",https://news.tvchosun.com/site/data/html_dir/2...
3,"미, 해상봉쇄 시작…트럼프 “이란, 간절히 합의 원해”",https://news.kbs.co.kr/news/pc/view/view.do?nc...
4,"정부, 이란에 한국 선박 정보 공유‥통항 협의 주목",https://imnews.imbc.com/news/2026/politics/art...
5,"밴스 미 부통령 ""1차 협상단은 권한 없었다⋯이제 공은 이란에""",http://mbn.mk.co.kr/pages/news/newsView.php?ca...
6,"[속보] 미-이란 협상 기대감에 코스피, 장중 6000선 재돌파",https://www.kmib.co.kr/article/view.asp?arcid=...
7,"트럼프 명령한 이란 항구 봉쇄·기뢰 제거…""최고 난이도 임무""",https://www.news1.kr/world/usa-canada/6135351
8,"이란 대통령 ""국제법 틀에서 협상, 성공 여부 美에 달려""",http://www.fnnews.com/news/202604140743268153
9,미·이란 “대화 계속”…봉쇄 속에도 협상 불씨 살아있다,http://www.edaily.co.kr/news/newspath.asp?news...
